# Newtonian Stellar Structure

## Why are neutron stars so interesting? 

### **Numerical investigation of neutron star structure**

Neutron stars are among the most compact objects in the universe. A star that contains more mass than our sun being compacted down into something about the size of a city will naturally have intriguing physics behind it.

This project is a personal deep dive into how gravity, pressure, and the equation of state of dense matter determine the structure of a neutron star.

---

**The key questions I wish to investigate are:**

1.) How does a star maintain its hydrostatic equilibrium?

2.) What changes when the density is extremely large?

3.) What changes in structure do we see when accounting for relativistic effects?

4.) Why does the maximum mass limit matter?

5.) How does the equation of state affect the mass-radius relationship? 

## 1. The Starting Model

Let's begin with a spherically symmetric, non-rotating star in hydrostatic equilibrium. The most idealistic and basic example.

The newtonian equations governing the structure of the star are:

$$(1) \quad \frac{dP}{dr} = -\frac{Gm(r)\rho(r)}{r^2}$$

and

$$(2) \quad \frac{dm}{dr} = 4\pi r^2\rho(r).$$

Where:
- $P(r)$ is pressure
- $\rho(r)$ is mass density
- $m(r)$ is mass enclosed within radius $r$
- $G$ is gravitational constant

The goal is to numerically integrate these equations outward from the center of the star until the pressure reaches zero.

---

**Brief discussion of equations and methodology:**

Eq. (1) essentially describes the relationship between pressure and the distance "$r$" from the center of the star. We know pressure inside the star must be greater in order to support all of the material above it against gravity, hence why we see the negative sign within the equation. As $r$ increases, Pressure, $P$, will decrease

Eq. (2) Describes the relationship between the distance $r$ and the enclosed mass located at that distance. Essentially: "as you move outward within this star, how fast are we accumulating mass?

We have 2 coupled differential equations in this instance. $\frac{dP}{dr}$ depends on $m$ and $\rho$. But we then see that $\rho$ depends on $P$ in our equation of state, and that m itself depends on $\rho$ as seen in Eq. (2). So essentially, we see that everything feeds into each other.

$P$ --> $\rho$ --> $m$ --> $P$

The lack of simple algebraic simplification means that we must integrate outwards. 

The code written will do this automatically without the need for manual calculation. It will stop once we reach an $r$ value where $P = 0$. We'll call this $"R"$ which represents the surface of the star.
At that point, we'll reach a point where the enclosed, accumulated mass is $M = m(R)$

---

## Code: Imports and Constants

In [3]:
# imports
import numpy as np
# import astropy as ap
import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp

In [7]:
# Defining constants
G = 6.674e-11          # m^3 kg^-1 s^-2
h = 6.626e-34          # J*s
hbar = h/(2*np.pi)     # J*s
c = 2.998e8            # m/s
M_sun = 1.988e30       # kg
R_sun = 6.957e8        # m
m_n = 1.675e-27        # Neutron mass - kg

# Conversions
km = 1e3               # m

### The Equation of State

The equations describing hydrostatic equilibrium, like (1) and (2), contain both the pressure $P$ and density $\rho$. In order to solve these equations, we need an additional relationship connecting these quantities. This relationship is what is referred to as an equation of state (EOS).

The EOS describes how the physical properties of a material are related under a particular set of conditions. For our stellar model, we write this relationship as

$$
P = P(\rho).
$$

The EOS is important because our hydrostatic equilibrium equation, Eq. (1), requires the density at every radius. Once the pressure is known, the EOS allows us to determine the corresponding density and continue integrating the stellar structure equations outward.

For this first model, we will use a **polytropic equation of state**:

$$
(3) \quad P = K\rho^\gamma.
$$

Note: this is **not** intended to be a realistic description of neutron-star matter. Instead, it provides a simple model that allows us to investigate the basic physics of stellar structure without introducing the complicated nuclear physics required for a realistic neutron-star equation of state.

The polytropic EOS is also convenient computationally because it can easily be inverted:

$$
(4) \quad \rho = \left(\frac{P}{K}\right)^{1/\gamma}.
$$

This allows the density to be calculated directly from the pressure during numerical integration. Later in the project, we can replace this simplified EOS with more realistic models and investigate how the properties of the resulting neutron stars change and how far these assumptions are from reality.

We will also assume that we are working with a standard monatomic gas, so we set out adiabatic index: $\gamma = \frac{5}/{3}$

We need to also determined our $K$ value, or our constant of proportionality.

The standard quantum mechanical expression for the pressure of a non-relativistic degenerate Fermi gas is:

$$
P = \frac{1}{5}(3\pi^2)^\frac{2}{3} \frac{\hbar}{m_n} n_n^\frac{5}{3}
$$

where $\hbar$ is the reduced Planck constant, $m_n$ is the mass of one neutron, and $n_n$ is the number density of neutrons (particles/$m^3$)

In order to get this to the form of our polytropic equation of state (Eq. 3), we sub in $n_n = \frac{\rho}{m_n}$

$$
P = \left[\frac{1}{5}(3\pi^2)^\frac{2}{3} \frac{\hbar}{m_n^\frac{8}{3}}\right]\rho^\frac{5}{3}
$$

where the section in the brackets is equal to $K$. Substituting $\hbar = \frac{h}{2\pi}$:

$$
K = \frac{1}{5}\left(\frac{3}{\pi}\right)^\frac{2}{3} \frac{h^2}{m_n^\frac{8}{3}}
$$

In [ ]:

# defining gamma and Solving for K:
gamma = 5/3
K = (1/5)*((3/np.pi)**(2/3))*((h**2)/(m_n**(8/3)))
print(f'the value of K is: {K:.3f}')

# Defining the polytropic EOS as a function:
def pressure(rho):
    return K * rho**gamma

# Defining equation 2 m(r) after integrating for dm = 4pi*r^2*rho dr from 0 to r_0:
def massR(rho_c, r0):
    return (4/3)*np.pi*r0**3*rho_c # Note that r0 is a very small radius close to the center

the value of K is: 21518.283


In [20]:
# Inverting the EOS:
def density(P):
    return (P / K)**(1/gamma)

# Writing the ODE system solver now:
def stellar_structure(r, y):
    P, m = y
    
    rho = density(P)
    
    dPdr = -G*m*rho/r**2
    dmdr = 4*np.pi*r**2*rho
    
    return [dPdr, dmdr]